In [1]:
from openai import OpenAI
import re
from dotenv import load_dotenv
_= load_dotenv()
client = OpenAI()

In [2]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

In [12]:
system_prompt = """
You are a system architect, you will be given output of the buggy code
you will run in a loop of Thought, action, PAUSE, Observation, at the end you output ANSWER code that fixes the bug along with what you tried to solve the bug
if the attempt solved the bug, ouput what caused the bug, what solved the bug, and how your solution solved the bug,
if the attempt didn't solve the bug, output what caused the bug and what solution you tried and why it failed along with the failing output.

available tools:

scan_repo: 
e.g. scan_repo: None
scans the default target repository and returns list that consists of all the file names the repository has.

read_file:
e.g. read_file: file_name.ext
reads all the contents of the file including jupyter notebook and returns string that consists of file content.

write_file: 
e.g. write_file: file_name.ext, content
writes content inside a file, this tool can be used to write the content inside a file.

record_attempt:
e.g. record_attempt: hypothesis, code_changed, result, why_it_failed
this is used to record all the attempts you done to solve the bug,
why_it_failed(argument): use this only when attempt has failed else pass None.

notebook_sandbox:
e.g. notebook_sanbox: 
it is a class that creates a jupyter notebook sandbox to experiment with the buggy code, it also has various tools:
    add_cell: code
    Appends a new Python code cell to the bottom of the virtual Jupyter notebook. Use this to write new experimental code or define functions. Returns the new cell's ID.
    execute_cell: cell_id
    Executes a specific code cell in the virtual notebook by its ID. Variables and functions defined here will persist in memory for future cells. Always check the output for runtime errors.
    edit_cell: cell_id, new_code
    Overwrites the Python code in an existing cell. Use this specifically to fix syntax or runtime errors in a cell that previously failed.
    get_sandbox_state: None
    Reads the entire virtual notebook. Returns a formatted string showing all current code cells, their IDs, and their last execution outputs. Use this if you lose track of the notebook's state or need to review past outputs.

execute_file:
e.g. execute_file: file_name.ext, timeout = 60
this is used to execute the file, returns string that contains output of the file executed,
timeout(argument): by default this is set to 60 secs but if you think the file needs more time to execute based on the contents of the file then you can modify this accordingly.

when you are about to create a python code to fix the bug you should write your code *strictly* in this format:

    <execute_python>
    # valid python code here
    </execute_python>
    
example session:

user_question: Application is crashing with a "TypeError: 'NoneType' object is not subscriptable" error.
Thought: I need to find the files in this repository.
Action: scan_repo:
PAUSE
you will be called again with:
Observation: ['data_processor.py']

Thought: user reported "TypeError: 'NoneType' object is not subscriptable" error. Given the repository structure, 'data_processor.py' is likely the file to perform a calculation and produce the error.
Action: read_file: data_processor.py
PAUSE

you will be called again with:
Observation: def get_average_score(scores):
    if not scores:
        return 0
    total = sum(scores)
    return total / len(scores)

def get_highest_score(scores):
    sorted_scores = scores.sort()
    return sorted_scores[-1]

if __name__ == "__main__":
    math_scores = [85, 92, 78, 99, 88]
    print(f"Average: {get_average_score(math_scores)}")
    print(f"Highest: {get_highest_score(math_scores)}")

Thought: I see the bug. I will create a notebook sandbox first.
Action: notebook_sandbox: 
PAUSE

you will be called again with:
Observation: notebook sandbox successfully created.

Thought: I will try method1: change scores.sort() to sort the list in-place and then return the last element. Let me test this logic in the sandbox first to ensure it outputs the correct highest score (99) without crashing.
Action: add_cell: [fix_code]
PAUSE

you will be called again with:
Observation: cell created with cell_id = [cell_id], content = [fix_code]

Thought: Now I will execute the cell.
Action: execute_cell: cell_id
PAUSE

you will be called again with:
Observation: output: [stdout]

Thought: Output confirms that the fix is working, I will reflect the changes back into the main file,
Action: write_file: [code]
PAUSE

you will be called again with:
Observation: file successfully changed.

then output:
Answer: bug is caused by [bug_code_part], fix that worked [fix_code], how it worked [hypothesis].
""".strip()

In [13]:
bot = Agent(system_prompt)

In [14]:
response = bot("Application that i built is crashing with a TypeError: 'NoneType' object is not subscriptable error")
response

'Thought: I need to find the files in this repository to identify where the error might be occurring.\nAction: scan_repo:\nPAUSE'

In [15]:
#files = scan_repo()
#next_prompt = f"Observation: {files}"
next_prompt = "observation: ['calculate_discount.py', 'test.ipynb', 'test_folder', 'test_folder/test_file.txt', data_processor.py]"
response = bot(next_prompt)
response

'Thought: The error "TypeError: \'NoneType\' object is not subscriptable" typically occurs when trying to access an element of a NoneType object as if it were a list or dictionary. Given the repository structure, \'calculate_discount.py\' and \'data_processor.py\' are likely candidates for containing the error. I will start by examining \'calculate_discount.py\' for potential issues.\nAction: read_file: calculate_discount.py\nPAUSE'

In [16]:
#file_content = read_file("calculate_discount.py")
next_prompt = """Observation: def calculate_discount(price, percent):
    # BUG: dividing by 10 instead of 100
    discount = price * percent / 10
    return price - discount


def apply_tax(price, tax_rate):
    return price * (1 + tax_rate)

print(calculate_discount(100, 10))""".strip()

response = bot(next_prompt)
response

'Thought: The `calculate_discount` function has a comment indicating a bug related to dividing by 10 instead of 100, but this wouldn\'t cause a "TypeError: \'NoneType\' object is not subscriptable" error. The error might be elsewhere. I will now check the \'data_processor.py\' file for potential issues.\nAction: read_file: data_processor.py\nPAUSE'

In [17]:
#file_content = read_file("data_processor.py")
next_prompt = """Observation: def get_average_score(scores):
    if not scores:
        return 0
    total = sum(scores)
    return total / len(scores)

def get_highest_score(scores):
    sorted_scores = scores.sort()
    return sorted_scores[-1]

if __name__ == "__main__":
    math_scores = [85, 92, 78, 99, 88]
    print(f"Average: {get_average_score(math_scores)}")
    print(f"Highest: {get_highest_score(math_scores)}")""".strip()

response = bot(next_prompt)
response

'Thought: In the `get_highest_score` function, `scores.sort()` sorts the list in place and returns `None`. Attempting to access `sorted_scores[-1]` results in a "TypeError: \'NoneType\' object is not subscriptable" error. I will fix this by sorting the list and then accessing the last element correctly.\n\nAction: notebook_sandbox:\nPAUSE'

In [18]:
next_prompt = "Observation: notebook sandbox successfully created."
response = bot(next_prompt)
response

'Thought: I will test the fix by modifying the `get_highest_score` function to sort the list and then access the last element correctly. I will do this in the notebook sandbox to ensure it outputs the correct highest score (99) without crashing.\n\nAction: add_cell: \n```python\ndef get_highest_score(scores):\n    scores.sort()\n    return scores[-1]\n\n# Test the function\nmath_scores = [85, 92, 78, 99, 88]\nprint(f"Highest: {get_highest_score(math_scores)}")\n```\nPAUSE'

In [19]:

def get_highest_score(scores):
    sorted_scores = sorted(scores)
    return sorted_scores[-1]

# Test the function
math_scores = [85, 92, 78, 99, 88]
get_highest_score(math_scores)

99

In [20]:
#i have to make my write_file tool capable of converting raw python code to well formatted one.
next_prompt = "Observation: 99"
response = bot(next_prompt)
response

'Thought: The output confirms that the fix is working correctly. The `get_highest_score` function now returns the correct highest score without causing a TypeError. I will now apply this fix to the main file.\n\nAction: write_file: data_processor.py, \n```python\ndef get_average_score(scores):\n    if not scores:\n        return 0\n    total = sum(scores)\n    return total / len(scores)\n\ndef get_highest_score(scores):\n    scores.sort()\n    return scores[-1]\n\nif __name__ == "__main__":\n    math_scores = [85, 92, 78, 99, 88]\n    print(f"Average: {get_average_score(math_scores)}")\n    print(f"Highest: {get_highest_score(math_scores)}")\n```\nPAUSE'

In [21]:
# def get_average_score(scores):
#     if not scores:
#         return 0
#     total = sum(scores)
#     return total / len(scores)

# def get_highest_score(scores):
#     scores.sort()
#     return scores[-1]

# if __name__ == "__main__":
#     math_scores = [85, 92, 78, 99, 88]
#     print(f"Average: {get_average_score(math_scores)}")
#     print(f"Highest: {get_highest_score(math_scores)}")
# LLM did successfully write the whole fixed code again, the question is if the code is really large, is it efficient to write the whole code again or just the fix? 
next_prompt = "Observation: data_processor.py successfully updated."
response = bot(next_prompt)
response

'Answer: The bug was caused by attempting to access an element of a `NoneType` object in the `get_highest_score` function. The `scores.sort()` method sorts the list in place and returns `None`, leading to the error when trying to access `sorted_scores[-1]`. The fix involved sorting the list in place and then accessing the last element directly from the `scores` list. This solution resolved the error by ensuring that the list is sorted before accessing its elements, thus avoiding the `NoneType` issue.'

In [ ]:
"""key observations:
1. prompt is working correctly, so no need to change it.
2. I need to modify my write_file tool to be able to extract and format the raw LLM python code.
3. LLM is writting the whole code again after the fix, which is fine for now but will it be able to generate 100s of lines of code if the file has those many code lines?,
4. At the end it is responding with a detailed report, but I have to format it properly.
5. I want the workflow to work in loop if one attempt fails it should try another attempt by looking at the methods it has already tried, try a fix read the output if fail, try again until success or max attempts is reached.
6. While automating these steps make sure if tool fails to return something, LLM is specified properly."""